In [ ]:
#@title Gene Expression Bootstrapper { display-mode: "form" }
from IPython.display import display, HTML
display(HTML("""
<style>
  .bio-details > summary { list-style: none; }
  .bio-details > summary::-webkit-details-marker { display: none; }
  .bio-details > summary::before {
    content: '\25bc';
    display: inline-block;
    margin-right: 0.6rem;
    font-size: 0.75rem;
    color: #888;
    transition: transform 0.15s ease;
  }
  .bio-details[open] > summary::before {
    transform: rotate(-180deg);
  }
</style>
<div style='font-family:sans-serif;'>
  <div style='display:flex;justify-content:space-between;align-items:flex-start;margin-bottom:1.5rem;'>
    <div style='flex:1;'>
      <h1 style='margin:0 0 0.5rem 0;font-size:2rem;font-weight:700;'>Gene Expression Bootstrapper</h1>
      <p style='margin:0;color:#444;font-size:16px;'>Bootstraps missing gene expression values for unmapped and unknown genes in metatranscriptomics data, sampling from genes in the same metabolic subsystem. Output files serve as the geneExpr folder input for FEA.</p>
    </div>
    <div style='margin-left:2rem;flex-shrink:0;'>
      <img src='https://raw.githubusercontent.com/MeMoModelling/proteinSeqMapping/main/bii_horizontal_logo_smalle472014210204e8886d5cf09c653e7e5.png' style='width:190px;'/>
    </div>
  </div>
  <hr style='border:none;border-top:1px solid #ddd;margin:1.5rem 0;'/>
  <h2 style='font-size:1.4rem;margin:0 0 0.8rem 0;'>Instructions:</h2>
  <ol style='margin:0 0 1.5rem 0;padding-left:1.5rem;font-size:16px;line-height:1.8;'>
    <li><strong>Optional:</strong> adjust the batch count in Step 2 (<strong>default: 1000 recommended</strong>).</li>
    <li>Go to <strong>Runtime &#8594; Run all</strong> in the top menu.</li>
    <li>Upload your files when prompted.</li>
    <li>The output ZIP will download automatically when complete.</li>
  </ol>

  <h3 style='font-size:1.1rem;margin:0 0 0.8rem 0;'>Inputs:</h3>

  <details class='bio-details' style='border:1.5px solid #e0e0e0;border-radius:8px;margin-bottom:0.6rem;background:#fafafa;'>
    <summary style='cursor:pointer;padding:0.75rem 1rem;display:flex;align-items:center;'>
      <span style='font-size:16px;font-weight:700;font-family:Arial,sans-serif;'>species_model.xlsx</span>
      <span style='font-size:16px;color:#666;margin-left:0.75rem;'>&#8212; Reactions sheet from the community assembly step. Must include <code>system</code> and <code>genes</code> columns. One file per species.</span>
    </summary>
    <div style='padding:0 1rem 0.75rem 1rem;'>
      <img src='https://raw.githubusercontent.com/MeMoModelling/Gene-Expression-Bootstrapper/main/preview_input_model.png' style='max-width:560px;border-radius:4px;border:1px solid #e0e0e0;'/>
    </div>
  </details>

  <details class='bio-details' style='border:1.5px solid #e0e0e0;border-radius:8px;margin-bottom:0.6rem;background:#fafafa;'>
    <summary style='cursor:pointer;padding:0.75rem 1rem;display:flex;align-items:center;'>
      <span style='font-size:16px;font-weight:700;font-family:Arial,sans-serif;'>species_mapping.xlsx</span>
      <span style='font-size:16px;color:#666;margin-left:0.75rem;'>&#8212; Model sheet from the gene annotation step. Maps model gene tags to genome annotation IDs. One file per species, in the same order as model files.</span>
    </summary>
    <div style='padding:0 1rem 0.75rem 1rem;'>
      <img src='https://raw.githubusercontent.com/MeMoModelling/Gene-Expression-Bootstrapper/main/preview_input_mapping.png' style='max-width:560px;border-radius:4px;border:1px solid #e0e0e0;'/>
    </div>
  </details>

  <details class='bio-details' style='border:1.5px solid #e0e0e0;border-radius:8px;margin-bottom:1.5rem;background:#fafafa;'>
    <summary style='cursor:pointer;padding:0.75rem 1rem;display:flex;align-items:center;'>
      <span style='font-size:16px;font-weight:700;font-family:Arial,sans-serif;'>combined_geneExpr.csv</span>
      <span style='font-size:16px;color:#666;margin-left:0.75rem;'>&#8212; Normalised gene expression counts for all species and samples. Rows are gene IDs, columns are sample names.</span>
    </summary>
    <div style='padding:0 1rem 0.75rem 1rem;'>
      <img src='https://raw.githubusercontent.com/MeMoModelling/Gene-Expression-Bootstrapper/main/preview_input_geneexpr.png' style='max-width:560px;border-radius:4px;border:1px solid #e0e0e0;'/>
    </div>
  </details>

  <h3 style='font-size:1.1rem;margin:0 0 0.8rem 0;'>Output:</h3>

  <details class='bio-details' style='border:1.5px solid #e0e0e0;border-radius:8px;background:#fafafa;'>
    <summary style='cursor:pointer;padding:0.75rem 1rem;display:flex;align-items:center;'>
      <span style='font-size:16px;font-weight:700;font-family:Arial,sans-serif;'>geneExpr_bootstrapped.zip</span>
      <span style='font-size:16px;color:#666;margin-left:0.75rem;'>&#8212; One CSV per bootstrap batch with expression values imputed for unmapped and unknown genes. Example of one file inside the ZIP:</span>
    </summary>
    <div style='padding:0 1rem 0.75rem 1rem;'>
      <img src='https://raw.githubusercontent.com/MeMoModelling/Gene-Expression-Bootstrapper/main/preview_output_geneexpr.png' style='max-width:560px;border-radius:4px;border:1px solid #e0e0e0;'/>
    </div>
  </details>

</div>
"""))

In [ ]:
#@title Step 1 — Set up environment { display-mode: "form" }
import subprocess, sys
from IPython.display import display, HTML

display(HTML("""
<div style='padding:0.8rem 1.2rem;background:#f0f4ff;border-left:4px solid #1a56db;border-radius:0 8px 8px 0;font-family:sans-serif;'>
  <b style='color:#1a56db;'>&#9881; Setting up environment...</b>
  <p style='margin:0.3rem 0 0 0;font-size:16px;color:#555;'>This runs automatically and takes about 1 minute.</p>
</div>
"""))

import os
if not os.path.isdir('/content/Gene-Expression-Bootstrapper'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/MeMoModelling/Gene-Expression-Bootstrapper.git',
                    '/content/Gene-Expression-Bootstrapper'], check=True, capture_output=True)
sys.path.insert(0, '/content/Gene-Expression-Bootstrapper')

subprocess.run(['pip', 'install', '-q', 'pandas', 'numpy', 'openpyxl', 'python-libsbml'], capture_output=True)

display(HTML("""
<div style='padding:0.8rem 1.2rem;background:#f0fff8;border-left:4px solid #006d5b;border-radius:0 8px 8px 0;font-family:sans-serif;'>
  <b style='color:#006d5b;'>&#10003; Ready! Proceed to the next step.</b>
</div>
"""))

In [ ]:
#@title Step 2 — Upload files & run { display-mode: "form" }
#@markdown ### Parameters (optional)
#@markdown **Default value works for most cases.** Only change if needed.
#@markdown
#@markdown | Parameter | Meaning |
#@markdown |---|---|
#@markdown | batch_count | Number of bootstrap iterations (= number of output CSV files) |
#@markdown
batch_count = 1000  #@param {type:"slider", min:100, max:5000, step:100}
#@markdown ---
#@markdown ### Upload your files
from IPython.display import display, HTML
from google.colab import files
import os, sys, zipfile

sys.path.insert(0, '/content/Gene-Expression-Bootstrapper')

display(HTML("""
<div style='padding:0.8rem 1.2rem;background:#f8f9fb;border:1.5px solid #d0d7e8;border-radius:8px;font-family:sans-serif;margin-bottom:0.8rem;'>
  <b style='font-size:16px;'>&#128193; Upload your prefixed model files</b>
  <p style='margin:0.3rem 0 0 0;font-size:16px;color:#555;'>One <code>.xlsx</code> per species &mdash; e.g. <code>species_model.xlsx</code></p>
</div>
"""))
uploaded_models = files.upload()
model_filenames = list(uploaded_models.keys())
display(HTML(f"<p style='font-family:sans-serif;font-size:16px;color:#006d5b;'>&#10003; Uploaded: {', '.join('<code>' + f + '</code>' for f in model_filenames)}</p>"))

display(HTML("""
<div style='padding:0.8rem 1.2rem;background:#f8f9fb;border:1.5px solid #d0d7e8;border-radius:8px;font-family:sans-serif;margin-bottom:0.8rem;'>
  <b style='font-size:16px;'>&#128193; Upload your identifier mapping files</b>
  <p style='margin:0.3rem 0 0 0;font-size:16px;color:#555;'>One <code>.xlsx</code> per species, <strong>in the same order</strong> as the model files &mdash; e.g. <code>species_mapping.xlsx</code></p>
</div>
"""))
uploaded_mappings = files.upload()
mapping_filenames = list(uploaded_mappings.keys())
display(HTML(f"<p style='font-family:sans-serif;font-size:16px;color:#006d5b;'>&#10003; Uploaded: {', '.join('<code>' + f + '</code>' for f in mapping_filenames)}</p>"))

display(HTML("""
<div style='padding:0.8rem 1.2rem;background:#f8f9fb;border:1.5px solid #d0d7e8;border-radius:8px;font-family:sans-serif;margin-bottom:0.8rem;'>
  <b style='font-size:16px;'>&#128193; Upload your combined gene expression CSV</b>
  <p style='margin:0.3rem 0 0 0;font-size:16px;color:#555;'><code>combined_geneExpr.csv</code></p>
</div>
"""))
uploaded_geneexpr = files.upload()
geneexpr_filename = list(uploaded_geneexpr.keys())[0]
display(HTML(f"<p style='font-family:sans-serif;font-size:16px;color:#006d5b;'>&#10003; Uploaded: <code>{geneexpr_filename}</code></p>"))

display(HTML("""
<div style='padding:0.8rem 1.2rem;background:#f0f4ff;border-left:4px solid #1a56db;border-radius:0 8px 8px 0;font-family:sans-serif;'>
  <b style='color:#1a56db;font-size:16px;'>&#8635; Processing&hellip; this may take several minutes depending on batch count.</b>
</div>
"""))

species_prefixes = [f.split('_')[0] for f in model_filenames]

from utils.bootstrap_genes import bootstrap_genes

output_dir = 'geneExpr_output'
os.makedirs(output_dir, exist_ok=True)

bootstrap_genes(
    model_pre_filenames=model_filenames,
    mapping_filenames=mapping_filenames,
    species_prefixes=species_prefixes,
    combined_geneExpr_filename=geneexpr_filename,
    geneExpr_folder=output_dir,
    batch_count=int(batch_count),
)

zip_name = 'geneExpr_bootstrapped.zip'
output_files = sorted([f for f in os.listdir(output_dir) if f.endswith('.csv')])
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in output_files:
        zf.write(os.path.join(output_dir, fname), arcname=fname)

display(HTML(f"""
<div style='padding:1rem 1.2rem;background:#f0fff8;border:1.5px solid #006d5b;border-radius:8px;font-family:sans-serif;margin-top:0.5rem;'>
  <b style='font-size:16px;color:#006d5b;'>&#10003; Done!</b>
  <p style='margin:0.4rem 0 0 0;font-size:16px;color:#333;'>{len(output_files)} bootstrapped files generated.</p>
  <p style='margin:0.4rem 0 0 0;font-size:16px;color:#333;'>Downloading <code>{zip_name}</code>&hellip;</p>
  <p style='margin:0.6rem 0 0 0;font-size:16px;color:#555;'>&#128193; If the download doesn't start automatically, find <code>{zip_name}</code> in the <b>Files panel on the left</b> and download manually.</p>
</div>
"""))

files.download(zip_name)